# Lesson 1: Claude Tool Use (Anthropic SDK)

This lesson reimplements OpenAI function calling with Claude's native tool use, using the Anthropic Python SDK directly (`client.messages.create(..., tools=[...])`).

In [ ]:
import os
import json
import anthropic
from dotenv import load_dotenv
load_dotenv()

# Default model is Haiku 4.5; override with e.g. LLM_MODEL=claude-opus-4-8
MODEL = os.environ.get("LLM_MODEL", "claude-haiku-4-5")

In [2]:
import json

def get_current_weather(location, unit="fahrenheit"):
    """Get the current weather in a given location"""
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

In [ ]:
# Anthropic tools use `input_schema` (JSON Schema) instead of OpenAI's `parameters`
tools = [{
    "name": "get_current_weather",
    "description": "get the current weather in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city and state, e.g. San Francisco, CA",
            },
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["location"],
    },
}
]

In [4]:
messages = [{
    "role": "user",
    "content": "what's the weather like in Boston?"
}]

In [ ]:
client = anthropic.Anthropic()
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print(response)

In [ ]:
# Claude returns a list of content blocks; a tool call is a `tool_use` block
response.content

In [ ]:
tool_use = next(block for block in response.content if block.type == "tool_use")
tool_use.name

In [ ]:
# tool_use.input is already a parsed dict (no json.loads needed)
args = tool_use.input
args

In [ ]:
get_current_weather(**args)

In [10]:
messages = [
    {
        "role": "user",
        "content": "hi!",
    }
]

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)
print(response)

In [ ]:
# tool_choice "auto": Claude decides whether to call a tool (the default)
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
    tool_choice={"type": "auto"},
)
print(response)

In [ ]:
# tool_choice "none": Claude cannot call a tool
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
    tool_choice={"type": "none"},
)
print(response)

In [ ]:
# Force a specific tool
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
    tool_choice={"type": "tool", "name": "get_current_weather"},
)
print(response)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What's the weather like in Boston!",
    }
]
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
    tool_choice={"type": "tool", "name": "get_current_weather"},
)
print(response)

In [ ]:
# Append the assistant turn (including its tool_use block) to the conversation
messages.append({"role": "assistant", "content": response.content})

In [ ]:
tool_use = next(block for block in response.content if block.type == "tool_use")
observation = get_current_weather(**tool_use.input)
observation

In [ ]:
# Return the tool result as a tool_result block, then let Claude answer
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use.id,
                "content": observation,
            }
        ],
    }
)

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print(response)